# NLPG: Exploring annotation in the NLTK Penn Treebank

This short exercise introduces several **representations of the same corpus data**:

- word tokens;
- sentences;
- part-of-speech (POS) tags;
- constituency parse trees;
- grammar productions extracted from those trees.

The main question is:

> **What becomes queryable when we add a new layer of annotation?**

There is nothing to submit. Work through the examples and keep brief notes on anything you find interesting or surprising.

## 1. Load the corpus

NLTK distributes a convenient sample of the Penn Treebank, consisting mainly of Wall Street Journal text.

The important point here is not its size, but that NLTK gives us **several aligned views of the same material**.

In [ ]:
import nltk
import os
from dotenv import load_dotenv

load_dotenv()
nltk.download("treebank", download_dir=os.environ["NLTK_DATA"])

from nltk.corpus import treebank
from collections import Counter

In [ ]:
print("Files:", len(treebank.fileids()))
print("Sentences:", len(treebank.sents()))
print("Word tokens:", len(treebank.words()))
print("First five file IDs:", treebank.fileids()[:5])

## 2. Tokens and sentences

Even this is not completely "raw" language: someone has already decided where sentence and token boundaries lie.

In [ ]:
sent = treebank.sents()[0]

print("As tokens:")
print(sent)

print("\nAs text:")
print(" ".join(sent))

### Activity 1

Change the sentence number below and inspect a few examples. (Some will still contain traces of annotation.)

Consider:

- What can you investigate from tokenised sentences alone?
- What grammatical questions are still difficult to ask?

In [ ]:
i = 10   # change this
print(" ".join(treebank.sents()[i]))

## 3. Add part-of-speech annotation

A POS-tagged sentence associates each token with a grammatical category.

In [ ]:
tagged = treebank.tagged_sents()[0]

for word, tag in tagged:
    print(f"{word:15} {tag}")

The Penn Treebank uses its own tag set. Examples include:

- `NN` — singular/common noun
- `NNS` — plural noun
- `NNP` — singular proper noun
- `VB`, `VBD`, `VBG`, ... — verb forms
- `JJ` — adjective
- `DT` — determiner

You do **not** need to memorise the tag set here.

In [ ]:
tag_counts = Counter(tag for _, tag in treebank.tagged_words())

print("Most frequent POS tags:")
for tag, count in tag_counts.most_common(15):
    print(f"{tag:8} {count}")

### Activity 2

POS annotation makes grammatical categories directly searchable.

Find some adjectives, then change `JJ` in your search to another tag such as `NN`, `NNS`, `VBD` or `NNP`.

In [ ]:
words_with_tag = [word for word, tag in treebank.tagged_words() if tag == "JJ"]

print(words_with_tag[:30])

## 4. Add constituency structure

The corpus also contains constituency parse trees.

A tree records **which words form constituents and how those constituents are nested**.

In [ ]:
tree = treebank.parsed_sents()[0]
print(tree)

In a local Jupyter notebook, `tree.draw()` can open a graphical window. In Colab, `pretty_print()` is more convenient.

In [ ]:
tree.pretty_print()

In [ ]:
print("Root label:", tree.label())

print("\nTop-level children:")
for child in tree:
    print(child)

### Activity 3

Search the corpus for noun phrases (`NP`), then try `VP` or `PP`.

This differs from ordinary text search: we are querying **structure**, not just words.

In [ ]:
shown = 0

for t in treebank.parsed_sents():
    for subtree in t.subtrees(lambda st: st.label() == "NP"):
        print(" ".join(subtree.leaves()))
        shown += 1
        if shown == 10:
            break
    if shown == 10:
        break

## 5. Trees contain grammar productions

A parsed tree can be decomposed into productions such as:

`S -> NP VP`

or

`NP -> DT NN`

These are the kinds of rules used in a context-free grammar.

In [ ]:
productions = tree.productions()

for p in productions[:20]:
    print(p)

In [ ]:
all_productions = []

for t in treebank.parsed_sents():
    all_productions.extend(t.productions())

production_counts = Counter(all_productions)

print("Most frequent productions:")
for production, count in production_counts.most_common(20):
    print(f"{count:5}  {production}")

### Activity 4 — A preview of PCFGs

Suppose we later want probabilities on grammar rules.

For every expansion whose left-hand side is `NP`, we can count how often each alternative occurs and normalise the counts.

In [ ]:
np_productions = [
    p for p in all_productions
    if str(p.lhs()) == "NP"
]

np_counts = Counter(np_productions)
total_np = sum(np_counts.values())

for production, count in np_counts.most_common(15):
    prob = count / total_np
    print(f"{prob:6.3f}   {count:5}   {production}")

These relative frequencies illustrate the basic idea behind estimating rule probabilities:

$$
P(\mathrm{RHS}\mid\mathrm{LHS})
=
\frac{\mathrm{count}(\mathrm{LHS}\rightarrow\mathrm{RHS})}
{\mathrm{count}(\mathrm{LHS}\rightarrow *)}
$$

We are **not doing PCFGs yet**. The point is simply to see where the empirical evidence can come from.

## 6. One corpus, several representations

| Representation | Example question |
|---|---|
| Tokens | How often does *market* occur? |
| Sentences | What words tend to occur together? |
| POS tags | Which nouns or adjectives occur? |
| Parse trees | Which noun phrases or structural patterns occur? |
| Productions | Which grammatical expansions occur, and how often? |

Each annotation layer makes new questions easier to ask.

It also embodies **analytical decisions** made by the annotators.

### Final small task

Choose one sentence number between 0 and 100. For the same sentence inspect:

1. its tokens;
2. its POS tags;
3. its parse tree.

Identify one fact that becomes visible only after POS annotation, and one that becomes visible only after constituency annotation.

In [ ]:
i = 25   # choose a sentence number

print("TOKENS")
print(treebank.sents()[i])

print("\nPOS TAGS")
print(treebank.tagged_sents()[i])

print("\nPARSE TREE")
treebank.parsed_sents()[i].pretty_print()

## Takeaway

A corpus is not just a collection of strings.

Different annotation layers preserve different kinds of information and therefore support different kinds of questions.

> **Annotation makes some generalisations queryable.**